# Brute Force Approach for Minimum fuel Trajectories in Earth Moon System

This notebook applies a brute force approach to solve the problem of launching a rocket from Low Earth Orbit (LEO) to Low Moon Orbit (LMO).

Two impulsive burns are appied. One at LEO and one at LMO.

The time of flight and phase of departure are also optimized

### Imports

In [1]:
import numpy as np
import pandas as pd
from cr3bp import (
    create_earth_moon_system,
    grid_search_method
)

### Initialize the System / Problem

In [2]:
# Create the Earth-Moon system using the cr3bp module
em = create_earth_moon_system()
print(em.info())

CR3BP System Information:
  Primary 1 mass: 5.972e+24 kg
  Primary 2 mass: 7.342e+22 kg
  Primary 1 radius: 6.371e+06 m
  Primary 2 radius: 1.737e+06 m
  Total mass: 6.045e+24 kg
  Distance: 3.844e+08 m (384400.0 km)
  Mass parameter μ: 0.012145

Characteristic scales:
  Length (l*): 3.844e+08 m (384400.0 km)
  Time (t*): 3.752e+05 s (4.343 days)
  Velocity (v*): 1.025e+03 m/s (1.025 km/s)
  Acceleration (a*): 2.731e-03 m/s^2
  Period: 27.285 days
None


In [3]:
# Define the LEO and LMO altitudes in meters
leo_alt_m=463e3
lmo_alt_m=100e3

## Run the Optimization Method

In [ ]:
dec_var_ranges = [[3.8, 4.1],[2.9, 3.5], [0.05, 0.15], [0.6, 0.9]]
#dec_var_ranges = new_ranges

In [5]:
results_df = grid_search_method(em, dec_var_ranges, 2.6e-4, leo_alt_m, lmo_alt_m)

Performing grid search with 10 x 10 x 5 x 10 = 5000 grid points...
Iteration: 2337/5000
Grid point satisfies constraint: Theta=3.93 rad, Delta_v=3.03, Delta_v_angle=0.12 rad, TOF=0.72 s => Distance to LMO=-16.69 km, Total Delta_v=20.09 km/s
New optimal found: Delta_v=3.11 km/s, Theta=3.93 rad, Delta_v_angle=0.12 rad, TOF=0.72 s, Distance to LMO=-16.69 km

Found 1 feasible solutions.


In [6]:
print(results_df)

      theta   delta_v  delta_v_angle       tof  total_delta_v  distance_to_lmo
0  3.928889  3.033333        0.12125  0.721111      19.612019        -0.000043


In [7]:
optimals = results_df.iloc[0][['theta', 'delta_v', 'delta_v_angle', 'tof']].values

In [8]:
np.save("grid_search_results.npy", optimals)
np.save("grid_search_results_df.npy", results_df)

In [9]:
shrink = 0.5
new_ranges = []
for i in range(4):
    current_span = dec_var_ranges[i][1] - dec_var_ranges[i][0]
    half_width = current_span * shrink / 2
    new_min = max(dec_var_ranges[i][0], optimals[i] - half_width)
    new_max = min(dec_var_ranges[i][1], optimals[i] + half_width)
    new_ranges.append([new_min, new_max])
print(new_ranges)

[[np.float64(3.923888888888889), np.float64(3.9338888888888888)], [np.float64(3.020833333333333), np.float64(3.0458333333333334)], [np.float64(0.1175), 0.125], [np.float64(0.716111111111111), np.float64(0.726111111111111)]]


In [10]:
results_df['distance_to_lmo'] = results_df['distance_to_lmo'] * em.l_star

In [11]:
print(results_df)

      theta   delta_v  delta_v_angle       tof  total_delta_v  distance_to_lmo
0  3.928889  3.033333        0.12125  0.721111      19.612019    -16687.451262
